# DESI DR1 LSS catalogue diagnostics

Interactive, reproducible diagnostics for the local DESI DR1 LSS research bundle. This notebook is intentionally descriptive: it does **not** estimate clustering, a density field, or voids, because those require DESI random catalogues and survey-mask treatment.

Run cells from top to bottom. The analysis scans the Parquet catalogue once, creates deterministic outputs in `figures/`, and displays the main results here.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display, Markdown

# Supports running from the repository root or from notebooks/.
ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / 'scripts'))
from analyze_desi_catalogue import (
    scan_catalogue, write_statistics_csv, plot_tracer_fractions, plot_cartesian_slice
)

INPUT = ROOT / 'data' / 'research' / 'desi_dr1_lss_research_bundle.parquet'
OUTPUT_DIR = ROOT / 'figures'
OUTPUT_DIR.mkdir(exist_ok=True)
assert INPUT.exists(), f'Research bundle not found: {INPUT}'
print(f'Repository root: {ROOT}')
print(f'Input bundle: {INPUT.name}')

In [ ]:
# Analysis controls — safe to adjust and rerun.
SLICE_Z_MPC = 0.0
SLICE_THICKNESS_MPC = 300.0
SLICE_RENDER_ROWS = 100_000
Z_MAX = 3.6
Z_BINS = 90
DPI = 240

In [ ]:
slice_frame, summary = scan_catalogue(
    INPUT,
    slice_z_mpc=SLICE_Z_MPC,
    slice_thickness_mpc=SLICE_THICKNESS_MPC,
    slice_render_rows=SLICE_RENDER_ROWS,
    z_max=Z_MAX,
    z_bins=Z_BINS,
)

statistics_path = OUTPUT_DIR / 'desi_dr1_tracer_statistics.csv'
composition_path = OUTPUT_DIR / 'desi_dr1_tracer_composition.png'
slice_path = OUTPUT_DIR / 'desi_dr1_cartesian_slice.png'

write_statistics_csv(summary, statistics_path)
plot_tracer_fractions(summary, composition_path, DPI)
plot_cartesian_slice(slice_frame, summary, slice_path, DPI)

print(f"Rows scanned: {summary['input_rows']:,}")
print(f"Deterministic slice rows rendered: {summary['cartesian_slice']['rendered_rows']:,}")
print(f"Saved: {statistics_path.relative_to(ROOT)}")
print(f"Saved: {composition_path.relative_to(ROOT)}")
print(f"Saved: {slice_path.relative_to(ROOT)}")

## Per-tracer catalogue summary

The coordinate residual compares Cartesian radius, $\sqrt{x^2+y^2+z^2}$, against the stored comoving distance. It is a coordinate-consistency diagnostic, not a cosmology fit.

In [ ]:
stats = (pd.DataFrame.from_dict(summary['tracer_statistics'], orient='index')
           .rename_axis('tracer')
           .reset_index())
stats['rows'] = stats['rows'].astype('int64')
display(stats.style.format({
    'rows': '{:,.0f}',
    'redshift_p16': '{:.4f}', 'redshift_median': '{:.4f}', 'redshift_p84': '{:.4f}',
    'coordinate_radius_minus_chi_mean_mpc': '{:.6g}',
    'coordinate_radius_minus_chi_rms_mpc': '{:.6g}',
    'coordinate_radius_minus_chi_mean_abs_mpc': '{:.6g}',
    'coordinate_radius_minus_chi_max_abs_mpc': '{:.6g}',
}))

## Tracer composition with redshift

This is the fraction of observed rows in each redshift bin. It is not completeness-corrected.

In [ ]:
display(Image(filename=str(composition_path)))

## Deterministic Cartesian slice

The plotted points are selected using the exact globally lowest stable object-ID hashes within the requested slice, so the same settings reproduce the same rendered sample. This is an observed-galaxy map, not a reconstructed density field.

In [ ]:
display(Image(filename=str(slice_path)))
display(slice_frame.head(10))

## Scientific boundary

To measure clustering, correlation functions, overdensity, or void candidates rigorously, add the tracer-specific DESI random catalogues, angular completeness/mask information, and a validated estimator.